# SwissFlakes Enterprise Data Exploration

Interactive exploration of the 4 enterprise data products:
- **Fulfillment** - order-to-delivery lifecycle
- **Customer 360** - unified customer view
- **Revenue Analytics** - revenue by shipping route
- **Compliance** - FINMA/BAZG/GwG transaction reporting

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()
print(f"Connected as: {session.get_current_user()}")
print(f"Database: {session.get_current_database()}")
print(f"Warehouse: {session.get_current_warehouse()}")

## Domain Overview

List all MART schemas and tables.

In [ ]:
schemas = session.sql("SHOW SCHEMAS IN DATABASE SFG_ENTERPRISE").to_pandas()
schemas[schemas["name"].str.startswith("MART")]

In [ ]:
for schema in ["MART_FULFILLMENT", "MART_CUSTOMER_360", "MART_REVENUE_ANALYTICS", "MART_COMPLIANCE"]:
    print(f"\n=== {schema} ===")
    try:
        tables = session.sql(f"SHOW TABLES IN SCHEMA SFG_ENTERPRISE.{schema}").to_pandas()
        if len(tables) > 0:
            print(tables[["name", "rows", "bytes"]].to_string())
        else:
            print("  (no tables yet - run dbt models)")
    except Exception as e:
        print(f"  Error: {e}")

## Fulfillment Lifecycle

Explore the order-to-delivery pipeline.

In [ ]:
try:
    fulfillment = session.sql("SELECT * FROM SFG_ENTERPRISE.MART_FULFILLMENT.FULFILLMENT_LIFECYCLE LIMIT 100").to_pandas()
    print(f"Rows: {len(fulfillment)}, Columns: {list(fulfillment.columns)}")
    fulfillment.head(10)
except Exception as e:
    print(f"Table not yet materialized: {e}")
    print("Run: EXECUTE DBT PROJECT SFG_ENTERPRISE.DCM.DBT_FULFILLMENT ARGS = 'run';")

## Revenue by Route

In [ ]:
try:
    revenue = session.sql("SELECT * FROM SFG_ENTERPRISE.MART_REVENUE_ANALYTICS.REVENUE_BY_ROUTE").to_pandas()
    print(f"Routes: {len(revenue)}")
    revenue.sort_values("TOTAL_REVENUE_CHF", ascending=False) if "TOTAL_REVENUE_CHF" in revenue.columns else revenue
except Exception as e:
    print(f"Table not yet materialized: {e}")

## Semantic Views

Check the 3 semantic views for Cortex Analyst.

In [ ]:
semantic_views = [
    "SFG_ENTERPRISE.MART_FULFILLMENT.FULFILLMENT_ANALYTICS",
    "SFG_ENTERPRISE.MART_REVENUE_ANALYTICS.REVENUE_ANALYTICS",
    "SFG_ENTERPRISE.MART_COMPLIANCE.COMPLIANCE_ANALYTICS",
]

for sv in semantic_views:
    print(f"\n=== {sv} ===")
    try:
        desc = session.sql(f"DESCRIBE SEMANTIC VIEW {sv}").to_pandas()
        print(desc.to_string())
    except Exception as e:
        print(f"  Not yet deployed: {e}")

## Source Data Products

Quick scan of source data.

In [ ]:
for db in ["SFG_LOGISTICS", "SFG_PAY"]:
    print(f"\n{'=' * 60}")
    print(f"  {db}")
    print(f"{'=' * 60}")
    try:
        schemas = session.sql(f"SHOW SCHEMAS IN DATABASE {db}").to_pandas()
        mart_schemas = schemas[schemas["name"].str.startswith("MART")]
        for _, row in mart_schemas.iterrows():
            schema_name = row["name"]
            tables = session.sql(f"SHOW TABLES IN SCHEMA {db}.{schema_name}").to_pandas()
            print(f"  {schema_name}: {len(tables)} table(s)")
    except Exception as e:
        print(f"  Error: {e}")